In [1]:
import yfinance as yf
import pandas as pd

In [2]:
raw = yf.download(
     ["KC=F", "BRL=X"],
    start="2018-12-31",
    end="2025-12-31",
    interval="1d"
)

df_market = raw["Close"].copy()
df_market.columns.name = None
df_market

[*********************100%***********************]  2 of 2 completed


,BRL=X,KC=F
Date,,
2018-12-31,3.8742,101.849998
2019-01-01,3.8800,NaN
2019-01-02,3.8799,99.500000
2019-01-03,3.7863,102.150002
2019-01-04,3.7551,101.599998
...,...,...
2025-12-23,5.5900,346.950012
2025-12-24,5.5185,345.149994
2025-12-26,5.5195,350.250000


In [3]:
# Renomeia e remove linhas sem preço do café (fins de semana, feriados da bolsa)
df_market = df_market.rename(columns={
    "KC=F": "preco_cafe",
    "BRL=X": "cambio_brl",
})


In [4]:
df_market.index = pd.to_datetime(df_market.index).tz_localize(None)
df_market.index.name = "date"
df_market = df_market[["preco_cafe", "cambio_brl"]].dropna(subset=["preco_cafe"])
df_market["cambio_brl"] = df_market["cambio_brl"].ffill()

df_market

,preco_cafe,cambio_brl
date,,
2018-12-31,101.849998,3.8742
2019-01-02,99.500000,3.8799
2019-01-03,102.150002,3.7863
2019-01-04,101.599998,3.7551
2019-01-07,102.750000,3.6612
...,...,...
2025-12-23,346.950012,5.5900
2025-12-24,345.149994,5.5185
2025-12-26,350.250000,5.5195


In [5]:
df_market.to_csv("../data/market_diario.csv", index=True)
print(f"Shape: {df_market.shape}")
df_market

Shape: (1763, 2)


,preco_cafe,cambio_brl
date,,
2018-12-31,101.849998,3.8742
2019-01-02,99.500000,3.8799
2019-01-03,102.150002,3.7863
2019-01-04,101.599998,3.7551
2019-01-07,102.750000,3.6612
...,...,...
2025-12-23,346.950012,5.5900
2025-12-24,345.149994,5.5185
2025-12-26,350.250000,5.5195


In [6]:
# \u2500\u2500 Merge clima \u00d7 mercado PRESERVANDO dias não-úteis \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# PROBLEMA do inner join direto: o KC=F só tem cotação em dias de pregão, mas o
# clima é diário. Um merge inner DESCARTA sábados, domingos e feriados — e com
# eles eventuais geadas ou chuvas de fim de semana, que são justamente o sinal
# de oferta que queremos. Uma geada de sábado simplesmente sumiria.
#
# SOLUÇÃO: cada dia de pregão recebe o clima ACUMULADO desde o pregão anterior
# (inclusive os dias não-úteis no intervalo). Assim, o clima de sex\u2192dom é
# carregado para a segunda-feira, com a estatística apropriada por variável.

df_market = pd.read_csv("../data/market_diario.csv", parse_dates=["date"])
df_climate = pd.read_csv("../data/inmet_wide_diario.csv", parse_dates=["data"])
df_climate = df_climate.rename(columns={"data": "date"}).sort_values("date")

# Dias de pregão (têm preço de café)
pregoes = df_market[["date"]].sort_values("date").reset_index(drop=True)

# Para cada dia climático, a qual pregão ele pertence?
# = o primeiro pregão >= a data climática (merge_asof "forward").
mapa = pd.merge_asof(
    df_climate.sort_values("date"),
    pregoes.assign(pregao=pregoes["date"]),
    on="date",
    direction="forward",
)
# Dias climáticos após o último pregão ficam sem destino \u2192 descarta
mapa = mapa.dropna(subset=["pregao"])

# Define a agregação por TIPO de variável (mesma lógica da agregação horária)
def tipo_agg(col: str) -> str:
    if col.startswith("precip_mm_sum") or col.startswith("radiacao"):
        return "sum"      # acumula chuva/radiação do bloco não-útil
    if col.startswith("temp_min") or col.startswith("graus_frio"):
        return "min" if col.startswith("temp_min") else "max"
    if col.startswith("temp_max"):
        return "max"
    if col.startswith("geada_risco"):
        return "max"      # houve risco de geada em ALGUM dia do bloco?
    return "mean"         # temperatura média, umidade, etc.

cols_clima = [c for c in mapa.columns if c not in ("date", "pregao")]
agg_map = {c: tipo_agg(c) for c in cols_clima}

clima_por_pregao = (mapa
                    .groupby("pregao")
                    .agg(agg_map)
                    .reset_index()
                    .rename(columns={"pregao": "date"}))

# Junta ao mercado (agora é seguro: 1 linha de clima por pregão)
df = df_market.merge(clima_por_pregao, on="date", how="left").sort_values("date").reset_index(drop=True)

# Pode haver pregão sem clima atribuído (raro). Preenche por interpolação temporal.
cols_c = [c for c in df.columns if c not in ("date", "preco_cafe", "cambio_brl")]
n_falta = df[cols_c].isnull().any(axis=1).sum()
if n_falta:
    print(f"{n_falta} pregão(ões) sem clima \u2192 interpolados no tempo.")
    df[cols_c] = df[cols_c].interpolate(method="linear", limit_area="inside").ffill().bfill()

df.to_csv("../data/dataset_final.csv", index=False)
print(f"Shape final: {df.shape}")
print(f"Período: {df['date'].min().date()} \u2192 {df['date'].max().date()}")
print(f"Dias com risco de geada no dataset final: "
      f"{int(df.filter(like='geada_risco').max(axis=1).sum())}")
df

1 pregão(ões) sem clima → interpolados no tempo.
Shape final: (1763, 47)
Período: 2018-12-31 → 2025-12-30
Dias com risco de geada no dataset final: 23


,date,preco_cafe,cambio_brl,geada_risco_Machado,geada_risco_Manhuacu,geada_risco_Patrocinio,geada_risco_Varginha,graus_frio_Machado,graus_frio_Manhuacu,graus_frio_Patrocinio,...,umidade_pct_mean_Patrocinio,umidade_pct_mean_Varginha,vento_rajada_ms_max_Machado,vento_rajada_ms_max_Manhuacu,vento_rajada_ms_max_Patrocinio,vento_rajada_ms_max_Varginha,vento_vel_ms_mean_Machado,vento_vel_ms_mean_Manhuacu,vento_vel_ms_mean_Patrocinio,vento_vel_ms_mean_Varginha
0,2018-12-31,101.849998,3.8742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,87.437500,77.000000,10.6,8.600000,11.600000,8.950000,1.222917,2.014583,1.381250,1.737500
1,2019-01-02,99.500000,3.8799,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,87.437500,77.000000,10.6,8.600000,11.600000,8.950000,1.222917,2.014583,1.381250,1.737500
2,2019-01-03,102.150002,3.7863,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,74.916667,80.208333,15.3,11.700000,8.900000,11.600000,1.716667,1.729167,1.225000,1.891667
3,2019-01-04,101.599998,3.7551,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,83.375000,91.291667,8.5,10.300000,8.300000,8.500000,2.087500,2.270833,0.633333,1.362500
4,2019-01-07,102.750000,3.6612,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,80.166667,83.838889,8.0,10.066667,7.933333,5.933333,1.519444,2.122222,1.087500,1.023056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1758,2025-12-23,346.950012,5.5900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,72.956522,78.000000,7.3,8.000000,7.900000,5.200000,1.662500,2.386957,1.100000,3.200000
1759,2025-12-24,345.149994,5.5185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,69.166667,78.000000,6.9,7.100000,6.700000,5.200000,1.125000,1.570833,0.733333,3.200000
1760,2025-12-26,350.250000,5.5195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,68.166667,78.000000,5.7,7.750000,5.800000,5.200000,1.312500,1.127083,0.589583,3.200000
1761,2025-12-29,352.149994,5.5425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,81.916667,78.000000,5.1,8.966667,7.433333,5.200000,0.725000,1.559722,0.754167,3.200000
